In [2]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
#import torch
import sklearn

In [2]:
dat = pd.read_csv('Data/cleaned_data.csv')
dat.columns

Index(['time', 'survey_date', 'last_4_digits_uid', 'last_name',
       'enrolled_course', 'major', 'minor', 'gender', 'gender_self_described',
       'ethnicity', 'first_gen_college', 'mother_education_level',
       'father_education_level', 'transfer_student', 'gpa_range', 'xp_courses',
       'xp_motivation', 'major_minor_motivation', 'belonging_rate',
       'hesitation_to_participate', 'respected_by_group',
       'perspective_inclusion', 'mistake_safety_in_group',
       'input_not_considered', 'comfort_asking_questions', 'meaningful_role',
       'non_valuable_contribution', 'community_work_valuable',
       'comfortable_sharing_ideas', 'feel_ignored',
       'community_partners_inclusion', 'community_partners_understanding',
       'lack_of_interaction_with_partners', 'do_patners_help', 'scenario_1',
       'scenario_1_reason', 'scenario_2', 'scenario_2_reason', 'scenario_3',
       'scenario_3_reason', 'scenario_4', 'scenario_4_reason', 'scenario_5',
       'scenario_5_reason'

In [3]:
import re

def clean_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

reason_columns = [
    'scenario_1_reason', 
    'scenario_2_reason', 
    'scenario_3_reason', 
    'scenario_4_reason', 
    'scenario_5_reason'
]

for col in reason_columns:
    dat[f'{col}_clean'] = dat[col].apply(clean_text)

print(dat[[f'{col}_clean' for col in reason_columns]].head())

                             scenario_1_reason_clean  \
0  Before acting directly on it I would go to the...   
1  I would definitely take a stand and respond to...   
2  School is very important, but I would feel a s...   
3  While it is important to acknowledge that acad...   
4  I would tell my partner to explain how they ar...   

                             scenario_2_reason_clean  \
0                I would be unsure what to say or do   
1  I would ask the instructor because they probab...   
2  I would like to get permission first to speak ...   
3  Because the community hasn't asked me to speak...   
4  if you want to speak up, you should help in an...   

                             scenario_3_reason_clean  \
0                I would take action. Ask questions.   
1  I would definitely consult the instructor. Giv...   
2  The results don’t have to necessarily be publi...   
3  Ethics, consent, and correct interpretation of...   
4  the organization is who you are serving, so

In [4]:
from transformers import pipeline
from tqdm import tqdm
tqdm.pandas()

sentiment_analyzer = pipeline(
    "sentiment-analysis", 
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True,
    max_length=512
)

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [5]:
def get_continuous_sentiment(text):
    if not text or pd.isna(text) or text.strip() == "":
        return None 
    try:
        result = sentiment_analyzer(text)[0]
        label = result['label'].lower()
        score = result['score']
        
        if label == 'positive':
            return score 
        elif label == 'negative':
            return -score
        else:
            return 0.0
            
    except Exception as e:
        return None

clean_columns = [
    'scenario_1_reason_clean', 
    'scenario_2_reason_clean', 
    'scenario_3_reason_clean', 
    'scenario_4_reason_clean', 
    'scenario_5_reason_clean'
]

for col in clean_columns:
    new_col_name = col.replace('_clean', '_sentiment')
    print(f"Processing {col}...")
    dat[new_col_name] = dat[col].progress_apply(get_continuous_sentiment)

sentiment_columns = [col.replace('_clean', '_sentiment') for col in clean_columns]
dat['overall_scenario_sentiment'] = dat[sentiment_columns].mean(axis=1)

Processing scenario_1_reason_clean...


100%|██████████| 45/45 [00:05<00:00,  8.72it/s]


Processing scenario_2_reason_clean...


100%|██████████| 45/45 [00:06<00:00,  7.12it/s]


Processing scenario_3_reason_clean...


100%|██████████| 45/45 [00:04<00:00, 11.12it/s]


Processing scenario_4_reason_clean...


100%|██████████| 45/45 [00:03<00:00, 12.37it/s]


Processing scenario_5_reason_clean...


100%|██████████| 45/45 [00:09<00:00,  4.52it/s]


In [6]:
dat.head()

,time,survey_date,last_4_digits_uid,last_name,enrolled_course,major,minor,gender,gender_self_described,ethnicity,...,scenario_2_reason_clean,scenario_3_reason_clean,scenario_4_reason_clean,scenario_5_reason_clean,scenario_1_reason_sentiment,scenario_2_reason_sentiment,scenario_3_reason_sentiment,scenario_4_reason_sentiment,scenario_5_reason_sentiment,overall_scenario_sentiment
0,1/29/2026 15:02:43,1/29/2026,532,Whitney,ELTS120XP,Global studies,"Food studies, global health",Man,NaN,White,...,I would be unsure what to say or do,I would take action. Ask questions.,"I’m not sure if I read this correctly, but it ...",I think a lot of barriers come up around findi...,0.0,-0.771393,0.0,0.575386,-0.608252,-0.160852
1,1/29/2026 22:54:03,1/29/2026,7824,Sleeper,ENGCOMP130DX,Public Affairs,"Professional Writing, Environmental Systems & ...",Woman,NaN,Hispanic/Latinx,...,I would ask the instructor because they probab...,I would definitely consult the instructor. Giv...,I would advise my friend to choose Job B. This...,I would say move the workshops to zoom. This w...,0.0,0.000000,0.0,0.696884,0.569326,0.253242
2,1/30/2026 12:26:40,1/30/2026,3402,Owen,CESC 191AX,Political Science,CESC,Woman,NaN,White,...,I would like to get permission first to speak ...,The results don’t have to necessarily be publi...,I would advise my friend that they may feel mo...,In-person meetings are much more valuable ways...,0.0,0.000000,0.0,0.826918,0.765378,0.318459
3,1/30/2026 13:24:01,1/30/2026,4689,Wenn,CESC191AX,Study of Religion,Community Engagement & Social Change,Woman,NaN,White,...,Because the community hasn't asked me to speak...,"Ethics, consent, and correct interpretation of...",I think the experience of working with both in...,Zoom meetings are more convenient than in-pers...,0.0,0.000000,0.0,0.915332,0.710048,0.325076
4,1/30/2026 13:30:49,1/30/2026,2809,Sobel,CESC 191AX,Political Science,CESC,Woman,NaN,White,...,"if you want to speak up, you should help in an...","the organization is who you are serving, so if...",More opportunities to expand the scope of the ...,Offer a hybrid option where families that cann...,0.0,0.000000,0.0,0.803777,0.000000,0.160755


In [8]:
dat.to_csv('Data/dat_withSentiment.csv', index=False)

In [9]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming your dataframe is named 'df' and you have already generated the sentiment columns
sentiment_cols = [
    'scenario_1_reason_sentiment', 'scenario_2_reason_sentiment', 
    'scenario_3_reason_sentiment', 'scenario_4_reason_sentiment', 
    'scenario_5_reason_sentiment'
]

# Variables you want to keep for grouping/x-axis
metadata_cols = ['enrolled_course', 'major', 'minor', 'gender', 'gender_self_described',
       'ethnicity', 'first_gen_college', 'mother_education_level',
       'father_education_level', 'transfer_student', 'gpa_range', 'xp_courses',
       'xp_motivation', 'major_minor_motivation', 'belonging_rate',
       'hesitation_to_participate', 'respected_by_group',
       'perspective_inclusion', 'mistake_safety_in_group',
       'input_not_considered', 'comfort_asking_questions', 'meaningful_role',
       'non_valuable_contribution', 'community_work_valuable',
       'comfortable_sharing_ideas', 'feel_ignored',
       'community_partners_inclusion', 'community_partners_understanding',
       'lack_of_interaction_with_partners', 'do_patners_help', 'scenario_1',
       'scenario_1_reason', 'scenario_2', 'scenario_2_reason', 'scenario_3',
       'scenario_3_reason', 'scenario_4', 'scenario_4_reason', 'scenario_5',
       'scenario_5_reason'] 

# Reshape the data
df_long = pd.melt(
    dat, 
    id_vars=metadata_cols, 
    value_vars=sentiment_cols,
    var_name='Scenario', 
    value_name='Sentiment_Score'
)

# Clean up the 'Scenario' names for the legend (e.g., "scenario_1_sentiment" -> "Scenario 1")
df_long['Scenario'] = df_long['Scenario'].str.replace('_sentiment', '').str.replace('_', ' ').str.title()

For plotting:
- faceted plots by course, gender, etc showing sentiment score distr with the scenario that connects to the other columns
- side by side bar plots?

In [3]:
## Do 1-2 really polished viz's for connection to community engagement
dat1 = pd.read_csv("Data/dat_withSentiment.csv")
dat1.head()

,time,survey_date,last_4_digits_uid,last_name,enrolled_course,major,minor,gender,gender_self_described,ethnicity,...,scenario_2_reason_clean,scenario_3_reason_clean,scenario_4_reason_clean,scenario_5_reason_clean,scenario_1_reason_sentiment,scenario_2_reason_sentiment,scenario_3_reason_sentiment,scenario_4_reason_sentiment,scenario_5_reason_sentiment,overall_scenario_sentiment
0,1/29/2026 15:02:43,1/29/2026,532,Whitney,ELTS120XP,Global studies,"Food studies, global health",Man,NaN,White,...,I would be unsure what to say or do,I would take action. Ask questions.,"I’m not sure if I read this correctly, but it ...",I think a lot of barriers come up around findi...,0.0,-0.771393,0.0,0.575386,-0.608252,-0.160852
1,1/29/2026 22:54:03,1/29/2026,7824,Sleeper,ENGCOMP130DX,Public Affairs,"Professional Writing, Environmental Systems & ...",Woman,NaN,Hispanic/Latinx,...,I would ask the instructor because they probab...,I would definitely consult the instructor. Giv...,I would advise my friend to choose Job B. This...,I would say move the workshops to zoom. This w...,0.0,0.000000,0.0,0.696884,0.569326,0.253242
2,1/30/2026 12:26:40,1/30/2026,3402,Owen,CESC 191AX,Political Science,CESC,Woman,NaN,White,...,I would like to get permission first to speak ...,The results don’t have to necessarily be publi...,I would advise my friend that they may feel mo...,In-person meetings are much more valuable ways...,0.0,0.000000,0.0,0.826918,0.765378,0.318459
3,1/30/2026 13:24:01,1/30/2026,4689,Wenn,CESC191AX,Study of Religion,Community Engagement & Social Change,Woman,NaN,White,...,Because the community hasn't asked me to speak...,"Ethics, consent, and correct interpretation of...",I think the experience of working with both in...,Zoom meetings are more convenient than in-pers...,0.0,0.000000,0.0,0.915332,0.710048,0.325076
4,1/30/2026 13:30:49,1/30/2026,2809,Sobel,CESC 191AX,Political Science,CESC,Woman,NaN,White,...,"if you want to speak up, you should help in an...","the organization is who you are serving, so if...",More opportunities to expand the scope of the ...,Offer a hybrid option where families that cann...,0.0,0.000000,0.0,0.803777,0.000000,0.160755


In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load data
df = pd.read_csv('Data/dat_withSentiment.csv')

# Connection heuristic items
pos_items = [
    'respected_by_group', 'perspective_inclusion', 'mistake_safety_in_group',
    'comfort_asking_questions', 'meaningful_role', 'community_work_valuable',
    'comfortable_sharing_ideas', 'community_partners_inclusion',
    'community_partners_understanding', 'do_patners_help'
]

rev_items = [
    'hesitation_to_participate', 'input_not_considered',
    'non_valuable_contribution', 'feel_ignored',
    'lack_of_interaction_with_partners'
]

for item in rev_items:
    df[item + '_rev'] = 6 - df[item]

all_heuristic_items = pos_items + [item + '_rev' for item in rev_items]
df['Connection_Score'] = df[all_heuristic_items].mean(axis=1)

sns.set_theme(style="whitegrid")

# Plot 1
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# 1st plot: Barplot
item_means = df[all_heuristic_items].mean().sort_values(ascending=True)
item_names = [name.replace('_rev', ' (Reversed)').replace('_', ' ').title() for name in item_means.index]

sns.barplot(x=item_means.values, y=item_names, ax=axes[0], palette="viridis")
axes[0].set_title('Mean Scores for Connection Heuristic Items', fontsize=16, pad=15)
axes[0].set_xlabel('Average Score (1 to 5)', fontsize=14)
axes[0].set_xlim(0, 5)

# 2nd plot: Swarm plot (hex-grid like points) for distribution instead of smooth KDE
sns.swarmplot(y=df['Connection_Score'], ax=axes[1], palette="viridis", marker="h", size=12)
# Also add a boxplot with stepped lines or just keep the swarm
sns.boxplot(y=df['Connection_Score'], ax=axes[1], color="white", linewidth=2, fliersize=0, boxprops=dict(alpha=0.3))
axes[1].set_title('Distribution of Overall Connection Heuristic Score', fontsize=16, pad=15)
axes[1].set_ylabel('Connection Score (1 to 5)', fontsize=14)
axes[1].set_ylim(2.8, 5.2)

plt.tight_layout()
plt.savefig('connection_heuristic_v3.png', dpi=300)
plt.close()


# Plot 2
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

df['scenario_2_short'] = df['scenario_2'].map({
    'I would first discuss the situation with my instructor or the community organization to seek guidance.': 'Discuss w/ Instructor',
    'I would take some form of action to ensure the community’s concerns are represented.': 'Take Action',
    'Given the circumstances, it would be better not to get involved or speak up.': 'Not Get Involved'
})

# Use swarmplot with hex markers instead of smooth violins
sns.boxplot(x='scenario_2_short', y='scenario_2_reason_sentiment', data=df, ax=axes[0], 
            color='white', linewidth=2, fliersize=0, boxprops=dict(alpha=0.5))
sns.swarmplot(x='scenario_2_short', y='scenario_2_reason_sentiment', data=df, ax=axes[0], 
              palette='viridis', marker='h', size=10, hue='scenario_2_short', legend=False)
axes[0].set_title('Sentiment vs. Scenario 2 Decisions\n(Community Concerns Representation)', fontsize=16, pad=15)
axes[0].set_xlabel('Decision Choice', fontsize=14)
axes[0].set_ylabel('Sentiment Score of Explanation', fontsize=14)

df['scenario_5_short'] = df['scenario_5'].map({
    'Move to zoom workshops': 'Move to Zoom',
    'Continue with in-person workshops': 'Continue In-Person'
})

sns.boxplot(x='scenario_5_short', y='scenario_5_reason_sentiment', data=df, ax=axes[1], 
            color='white', linewidth=2, fliersize=0, boxprops=dict(alpha=0.5))
sns.swarmplot(x='scenario_5_short', y='scenario_5_reason_sentiment', data=df, ax=axes[1], 
              palette='viridis', marker='h', size=10, hue='scenario_5_short', legend=False)
axes[1].set_title('Sentiment vs. Scenario 5 Decisions\n(Workshop Community Accessibility)', fontsize=16, pad=15)
axes[1].set_xlabel('Decision Choice', fontsize=14)
axes[1].set_ylabel('Sentiment Score of Explanation', fontsize=14)

plt.tight_layout()
plt.savefig('scenario_sentiments_v3.png', dpi=300)
plt.close()

/var/folders/cw/cx368b590mb_cg5tg17mvqv40000gn/T/ipykernel_11703/115380370.py:38: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=item_means.values, y=item_names, ax=axes[0], palette="viridis")
/var/folders/cw/cx368b590mb_cg5tg17mvqv40000gn/T/ipykernel_11703/115380370.py:44: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.swarmplot(y=df['Connection_Score'], ax=axes[1], palette="viridis", marker="h", size=12)
/opt/anaconda3/lib/python3.12/site-packages/seaborn/categorical.py:3399: UserWarning: 44.0% of the points cannot be placed; you may want to decrease the size of the markers or use stripplot.
  warnings.warn(msg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packages/seaborn/categorical.py:3399

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import textwrap

# Load data
df = pd.read_csv('Data/dat_withSentiment.csv')

# Connection heuristic items
pos_items = [
    'respected_by_group', 'perspective_inclusion', 'mistake_safety_in_group',
    'comfort_asking_questions', 'meaningful_role', 'community_work_valuable',
    'comfortable_sharing_ideas', 'community_partners_inclusion',
    'community_partners_understanding', 'do_patners_help'
]

rev_items = [
    'hesitation_to_participate', 'input_not_considered',
    'non_valuable_contribution', 'feel_ignored',
    'lack_of_interaction_with_partners'
]

for item in rev_items:
    df[item + '_rev'] = 6 - df[item]

all_heuristic_items = pos_items + [item + '_rev' for item in rev_items]
df['Connection_Score'] = df[all_heuristic_items].mean(axis=1)

# Map Scenarios
df['scenario_2_short'] = df['scenario_2'].map({
    'I would first discuss the situation with my instructor or the community organization to seek guidance.': 'Discuss w/ Instructor',
    'I would take some form of action to ensure the community’s concerns are represented.': 'Take Action',
    'Given the circumstances, it would be better not to get involved or speak up.': 'Not Get Involved'
})

df['scenario_5_short'] = df['scenario_5'].map({
    'Move to zoom workshops': 'Move to Zoom',
    'Continue with in-person workshops': 'Continue In-Person'
})

sns.set_theme(style="whitegrid")

# Helper function to wrap text for labels
def wrap_labels(ax, width, break_long_words=False):
    labels = []
    for label in ax.get_xticklabels():
        text = label.get_text()
        labels.append(textwrap.fill(text, width=width, break_long_words=break_long_words))
    ax.set_xticklabels(labels, rotation=45, ha='right')

facet_vars = ['gender', 'ethnicity', 'enrolled_course']
titles = ['Gender', 'Ethnicity', 'Enrolled Course']

for var, title in zip(facet_vars, titles):
    # Fill NaN with 'Unknown' for plotting
    df[var] = df[var].fillna('Unknown')
    
    fig, axes = plt.subplots(1, 3, figsize=(22, 8))
    
    # 1. Connection Score Distribution
    sns.boxplot(x=var, y='Connection_Score', data=df, ax=axes[0],
                color='white', linewidth=2, fliersize=0, boxprops=dict(alpha=0.5))
    sns.swarmplot(x=var, y='Connection_Score', data=df, ax=axes[0],
                  palette='viridis', marker='h', size=8, hue=var, legend=False)
    axes[0].set_title(f'Connection Score by {title}', fontsize=14, pad=15)
    axes[0].set_ylabel('Connection Score (1 to 5)')
    axes[0].set_xlabel('')
    wrap_labels(axes[0], 15)
    
    # 2. Scenario 2
    sns.boxplot(x=var, y='scenario_2_reason_sentiment', hue='scenario_2_short', data=df, ax=axes[1],
                color='white', linewidth=1, fliersize=0, boxprops=dict(alpha=0.5))
    sns.swarmplot(x=var, y='scenario_2_reason_sentiment', hue='scenario_2_short', data=df, ax=axes[1],
                  palette='viridis', marker='h', size=7, dodge=True)
    axes[1].set_title(f'Scenario 2 Sentiment by {title}', fontsize=14, pad=15)
    axes[1].set_ylabel('Sentiment Score')
    axes[1].set_xlabel('')
    axes[1].legend(title='Scenario 2 Decision', bbox_to_anchor=(1.05, 1), loc='upper left')
    wrap_labels(axes[1], 15)

    # 3. Scenario 5
    sns.boxplot(x=var, y='scenario_5_reason_sentiment', hue='scenario_5_short', data=df, ax=axes[2],
                color='white', linewidth=1, fliersize=0, boxprops=dict(alpha=0.5))
    sns.swarmplot(x=var, y='scenario_5_reason_sentiment', hue='scenario_5_short', data=df, ax=axes[2],
                  palette='viridis', marker='h', size=7, dodge=True)
    axes[2].set_title(f'Scenario 5 Sentiment by {title}', fontsize=14, pad=15)
    axes[2].set_ylabel('Sentiment Score')
    axes[2].set_xlabel('')
    axes[2].legend(title='Scenario 5 Decision', bbox_to_anchor=(1.05, 1), loc='upper left')
    wrap_labels(axes[2], 15)
    
    plt.tight_layout()
    plt.savefig(f'faceted_{var}.png', dpi=300)
    plt.close()

print("Faceted plots generated successfully.")

/var/folders/cw/cx368b590mb_cg5tg17mvqv40000gn/T/ipykernel_11703/2876797516.py:50: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(labels, rotation=45, ha='right')
/var/folders/cw/cx368b590mb_cg5tg17mvqv40000gn/T/ipykernel_11703/2876797516.py:72: FutureWarning: 

Setting a gradient palette using color= is deprecated and will be removed in v0.14.0. Set `palette='dark:white'` for the same effect.

  sns.boxplot(x=var, y='scenario_2_reason_sentiment', hue='scenario_2_short', data=df, ax=axes[1],
/opt/anaconda3/lib/python3.12/site-packages/seaborn/categorical.py:3399: UserWarning: 50.0% of the points cannot be placed; you may want to decrease the size of the markers or use stripplot.
  warnings.warn(msg, UserWarning)
/var/folders/cw/cx368b590mb_cg5tg17mvqv40000gn/T/ipykernel_11703/2876797516.py:50: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after se

Faceted plots generated successfully.


In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import textwrap

# Load data
df = pd.read_csv('Data/dat_withSentiment.csv')

# Connection heuristic items
pos_items = [
    'respected_by_group', 'perspective_inclusion', 'mistake_safety_in_group',
    'comfort_asking_questions', 'meaningful_role', 'community_work_valuable',
    'comfortable_sharing_ideas', 'community_partners_inclusion',
    'community_partners_understanding', 'do_patners_help'
]

rev_items = [
    'hesitation_to_participate', 'input_not_considered',
    'non_valuable_contribution', 'feel_ignored',
    'lack_of_interaction_with_partners'
]

for item in rev_items:
    df[item + '_rev'] = 6 - df[item]

all_heuristic_items = pos_items + [item + '_rev' for item in rev_items]
df['Connection_Score'] = df[all_heuristic_items].mean(axis=1)

sns.set_theme(style="whitegrid")

# Figure 1: Course Engagement (Resource Allocation focus)
fig, ax = plt.subplots(figsize=(14, 7))
course_avg = df.groupby('enrolled_course')['Connection_Score'].mean().sort_values()

# Clean course names for display if needed
course_labels = [textwrap.fill(str(c), 25) for c in course_avg.index]

sns.barplot(x=course_avg.values, y=course_labels, palette="viridis", ax=ax)
ax.set_title('Average Community Connection Score by Course\n(Identifying which programs need more support and resources)', fontsize=18, pad=15, weight='bold')
ax.set_xlabel('Average Connection Score (Scale: 1 to 5)', fontsize=14)
ax.set_ylabel('')
ax.set_xlim(3.0, 5.0) # Zooming in on the 3-5 range since all scores are high

# Add value labels
for p in ax.patches:
    ax.annotate(f"{p.get_width():.2f}", 
                (p.get_width() + 0.03, p.get_y() + p.get_height() / 2.), 
                ha='left', va='center', fontsize=12, color='black', weight='bold')

plt.tight_layout()
plt.savefig('course_resource_allocation.png', dpi=300)
plt.close()


# Figure 2: Equity in Engagement (Demographics Focus)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# First-Gen
fg_avg = df.groupby('first_gen_college')['Connection_Score'].mean().dropna().sort_values(ascending=False)
sns.barplot(x=fg_avg.index, y=fg_avg.values, palette="viridis", ax=axes[0])
axes[0].set_title('Connection by First-Generation Status\n(Targeting support for First-Gen students)', fontsize=16, weight='bold', pad=10)
axes[0].set_ylabel('Average Connection Score (Scale: 1 to 5)', fontsize=14)
axes[0].set_xlabel('First Generation College Student?', fontsize=14)
axes[0].set_ylim(3.0, 5.0)

for p in axes[0].patches:
    axes[0].annotate(f"{p.get_height():.2f}", 
                     (p.get_x() + p.get_width() / 2., p.get_height() + 0.05), 
                     ha='center', va='center', fontsize=14, weight='bold')

# Ethnicity
eth_avg = df.groupby('ethnicity')['Connection_Score'].mean().dropna().sort_values(ascending=False)
eth_labels = [textwrap.fill(str(l), 15) for l in eth_avg.index]

sns.barplot(x=eth_labels, y=eth_avg.values, palette="viridis", ax=axes[1])
axes[1].set_title('Connection by Ethnicity\n(Identifying equity gaps across student groups)', fontsize=16, weight='bold', pad=10)
axes[1].set_ylabel('')
axes[1].set_xlabel('')
axes[1].set_ylim(3.0, 5.0)
axes[1].tick_params(axis='x', labelsize=12)

for p in axes[1].patches:
    axes[1].annotate(f"{p.get_height():.2f}", 
                     (p.get_x() + p.get_width() / 2., p.get_height() + 0.05), 
                     ha='center', va='center', fontsize=14, weight='bold')

plt.suptitle('Equity and Inclusion: Where Should Targeted Resources Go?', fontsize=20, weight='bold')
plt.tight_layout()
plt.savefig('equity_resource_allocation.png', dpi=300)
plt.close()


# Figure 3: Scenario Action Propensity (Civic Responsibility Focus)
# Simplifying scenario outcomes to binary/ternary categories
df['scen2_action'] = df['scenario_2'].map({
    'I would first discuss the situation with my instructor or the community organization to seek guidance.': 'Seek Guidance\n(Instructor/Partner)',
    'I would take some form of action to ensure the community’s concerns are represented.': 'Direct Action\n(Advocate)',
    'Given the circumstances, it would be better not to get involved or speak up.': 'Not Get Involved\n(Passive)'
})
scen2_counts = df['scen2_action'].value_counts(normalize=True) * 100

df['scen5_action'] = df['scenario_5'].map({
    'Move to zoom workshops': 'Move to Zoom\n(Prioritize Accessibility)',
    'Continue with in-person workshops': 'Continue In-Person\n(Maintain Format)'
})
scen5_counts = df['scen5_action'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

sns.barplot(x=scen2_counts.index, y=scen2_counts.values, palette="viridis", ax=axes[0])
axes[0].set_title('Scenario 2: How Students Respond to Community Concerns', fontsize=16, weight='bold', pad=10)
axes[0].set_ylabel('Percentage of Students (%)', fontsize=14)
axes[0].set_xlabel('')
axes[0].set_ylim(0, 100)
axes[0].tick_params(axis='x', labelsize=12)

for p in axes[0].patches:
    axes[0].annotate(f"{p.get_height():.1f}%", 
                     (p.get_x() + p.get_width() / 2., p.get_height() + 3), 
                     ha='center', va='center', fontsize=14, weight='bold')
                     
sns.barplot(x=scen5_counts.index, y=scen5_counts.values, palette="viridis", ax=axes[1])
axes[1].set_title('Scenario 5: Prioritizing Workshop Accessibility', fontsize=16, weight='bold', pad=10)
axes[1].set_ylabel('')
axes[1].set_xlabel('')
axes[1].set_ylim(0, 100)
axes[1].tick_params(axis='x', labelsize=12)

for p in axes[1].patches:
    axes[1].annotate(f"{p.get_height():.1f}%", 
                     (p.get_x() + p.get_width() / 2., p.get_height() + 3), 
                     ha='center', va='center', fontsize=14, weight='bold')

plt.suptitle('Student Readiness for Active Community Advocacy\n(Demonstrating ROI on Civic Engagement Training)', fontsize=20, weight='bold')
plt.tight_layout()
plt.savefig('advocacy_readiness.png', dpi=300)
plt.close()

/var/folders/cw/cx368b590mb_cg5tg17mvqv40000gn/T/ipykernel_11703/2736165878.py:39: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=course_avg.values, y=course_labels, palette="viridis", ax=ax)
/var/folders/cw/cx368b590mb_cg5tg17mvqv40000gn/T/ipykernel_11703/2736165878.py:61: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=fg_avg.index, y=fg_avg.values, palette="viridis", ax=axes[0])
/var/folders/cw/cx368b590mb_cg5tg17mvqv40000gn/T/ipykernel_11703/2736165878.py:76: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=eth_labels, y=eth_avg.values, 

In [16]:
import textwrap
import random
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv('Data/dat_withSentiment.csv')

# Connection heuristic items explicitly mapped to Community Engagement
pos_items = [
    'respected_by_group', 'perspective_inclusion', 'mistake_safety_in_group',
    'comfort_asking_questions', 'meaningful_role', 'community_work_valuable',
    'comfortable_sharing_ideas', 'community_partners_inclusion',
    'community_partners_understanding', 'do_patners_help'
]
rev_items = [
    'hesitation_to_participate', 'input_not_considered',
    'non_valuable_contribution', 'feel_ignored',
    'lack_of_interaction_with_partners'
]

for item in rev_items:
    df[item + '_rev'] = 6 - df[item]

all_heuristic_items = pos_items + [item + '_rev' for item in rev_items]
df['CE_Connection_Score'] = df[all_heuristic_items].mean(axis=1)

# Ensure overall sentiment is calculated
sentiment_cols = [c for c in df.columns if 'sentiment' in c and c != 'overall_scenario_sentiment']
df['overall_scenario_sentiment'] = df[sentiment_cols].mean(axis=1)

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(20, 16))

# Plot 1: Course Quadrant
ax1 = axes[0, 0]
course_data = df.groupby('enrolled_course')[['CE_Connection_Score', 'overall_scenario_sentiment']].mean().dropna()

sns.scatterplot(data=course_data, x='CE_Connection_Score', y='overall_scenario_sentiment', 
                s=250, color=sns.color_palette("viridis", 1)[0], ax=ax1, edgecolor='black')

# Quadrant lines
med_conn_course = course_data['CE_Connection_Score'].median()
med_sent_course = course_data['overall_scenario_sentiment'].median()
ax1.axvline(med_conn_course, color='gray', linestyle='--', alpha=0.5)
ax1.axhline(med_sent_course, color='gray', linestyle='--', alpha=0.5)

# Annotations
for course in course_data.index:
    # Generate a random offset for each label
    x_offset = random.randint(5, 15)
    y_offset = random.randint(-15, 15)
    
    ax1.annotate(textwrap.fill(str(course), 15), 
                 (course_data.loc[course, 'CE_Connection_Score'], course_data.loc[course, 'overall_scenario_sentiment']),
                 xytext=(x_offset, y_offset), 
                 textcoords='offset points', 
                 fontsize=12, 
                 weight='bold',
                 arrowprops=dict(arrowstyle="-", color='gray', alpha=0.5))

ax1.set_title('1. Course Matrix: Community Engagement', fontsize=16, weight='bold')
ax1.set_xlabel('Community Engagement Connection Score', fontsize=12)
ax1.set_ylabel('Scenario Sentiment Score', fontsize=12)

# Plot 2: Ethnicity Quadrant
ax2 = axes[0, 1]
eth_data = df.groupby('ethnicity')[['CE_Connection_Score', 'overall_scenario_sentiment']].mean().dropna()
# Add count for bubble size
eth_data['count'] = df['ethnicity'].value_counts()

sns.scatterplot(data=eth_data, x='CE_Connection_Score', y='overall_scenario_sentiment', 
                size='count', sizes=(150, 900), color=sns.color_palette("viridis", 4)[2], ax=ax2, legend=False, edgecolor='black')

med_conn_eth = eth_data['CE_Connection_Score'].median()
med_sent_eth = eth_data['overall_scenario_sentiment'].median()
ax2.axvline(med_conn_eth, color='gray', linestyle='--', alpha=0.5)
ax2.axhline(med_sent_eth, color='gray', linestyle='--', alpha=0.5)

for eth in eth_data.index:
    ax2.annotate(textwrap.fill(str(eth), 15), 
                 (eth_data.loc[eth, 'CE_Connection_Score'], eth_data.loc[eth, 'overall_scenario_sentiment']),
                 xytext=(10, 10), textcoords='offset points', fontsize=12, weight='bold')

ax2.set_title('2. Equity Matrix: Engagement by Ethnicity', fontsize=16, weight='bold')
ax2.set_xlabel('Community Engagement Connection Score', fontsize=12)
ax2.set_ylabel('Scenario Sentiment Score', fontsize=12)

# Plot 3: First-Gen Bar Chart (Dual Axis)
ax3 = axes[1, 0]
fg_data = df.groupby('first_gen_college')[['CE_Connection_Score', 'overall_scenario_sentiment']].mean().dropna()

x = np.arange(len(fg_data.index))
width = 0.35

ax3_twin = ax3.twinx()
bar1 = ax3.bar(x - width/2, fg_data['CE_Connection_Score'], width, label='Engagement Connection Score', color=sns.color_palette("viridis", 5)[0])
bar2 = ax3_twin.bar(x + width/2, fg_data['overall_scenario_sentiment'], width, label='Sentiment Score', color=sns.color_palette("viridis", 5)[3])

ax3.set_ylabel('Engagement Connection Score (1-5)', fontsize=12)
ax3_twin.set_ylabel('Sentiment Score', fontsize=12)
ax3.set_xticks(x)
ax3.set_xticklabels(fg_data.index, fontsize=13, weight='bold')
ax3.set_title('3. First-Generation Student Engagement Gap', fontsize=16, weight='bold')

lines, labels = ax3.get_legend_handles_labels()
lines2, labels2 = ax3_twin.get_legend_handles_labels()
ax3_twin.legend(lines + lines2, labels + labels2, loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=2, fontsize=11)
ax3.set_ylim(3.5, 5.0)

# Plot 4: Transfer Student Bar Chart (Dual Axis)
ax4 = axes[1, 1]
ts_data = df.groupby('transfer_student')[['CE_Connection_Score', 'overall_scenario_sentiment']].mean().dropna()

x2 = np.arange(len(ts_data.index))

ax4_twin = ax4.twinx()
bar3 = ax4.bar(x2 - width/2, ts_data['CE_Connection_Score'], width, label='Engagement Connection Score', color=sns.color_palette("viridis", 5)[0])
bar4 = ax4_twin.bar(x2 + width/2, ts_data['overall_scenario_sentiment'], width, label='Sentiment Score', color=sns.color_palette("viridis", 5)[3])

ax4.set_ylabel('Engagement Connection Score (1-5)', fontsize=12)
ax4_twin.set_ylabel('Sentiment Score', fontsize=12)
ax4.set_xticks(x2)
ax4.set_xticklabels(ts_data.index, fontsize=13, weight='bold')
ax4.set_title('4. Transfer Student Engagement Gap', fontsize=16, weight='bold')

lines3, labels3 = ax4.get_legend_handles_labels()
lines4, labels4 = ax4_twin.get_legend_handles_labels()
ax4_twin.legend(lines3 + lines4, labels3 + labels4, loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=2, fontsize=11)
ax4.set_ylim(3.5, 5.0)

fig.suptitle("Student Sentiment against Core Engagement Metrics (Belonging, Meaningful Roles, Partner Collaboration)", ha='center', fontsize=26, color='black')

plt.tight_layout(rect=[0, 0.03, 1, 0.93])
plt.savefig('community_engagement_resource_allocation.png', dpi=300)
plt.close()

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import textwrap

# Load data
df = pd.read_csv('Data/dat_withSentiment.csv')

# Connection heuristic items explicitly mapped to Community Engagement
pos_items = [
    'respected_by_group', 'perspective_inclusion', 'mistake_safety_in_group',
    'comfort_asking_questions', 'meaningful_role', 'community_work_valuable',
    'comfortable_sharing_ideas', 'community_partners_inclusion',
    'community_partners_understanding', 'do_patners_help'
]
rev_items = [
    'hesitation_to_participate', 'input_not_considered',
    'non_valuable_contribution', 'feel_ignored',
    'lack_of_interaction_with_partners'
]

# Reverse the negative items (assuming 1-5 scale)
for item in rev_items:
    df[item + '_rev'] = 6 - df[item]

all_heuristic_items = pos_items + [item + '_rev' for item in rev_items]
df['CE_Connection_Score'] = df[all_heuristic_items].mean(axis=1)

# Clean up major names and handle missing values
df['major'] = df['major'].fillna('Unknown')
df['major'] = df['major'].str.title()

# Map Scenarios to shorter labels
df['scenario_2_short'] = df['scenario_2'].map({
    'I would first discuss the situation with my instructor or the community organization to seek guidance.': 'Discuss w/ Instructor',
    'I would take some form of action to ensure the community’s concerns are represented.': 'Take Action',
    'Given the circumstances, it would be better not to get involved or speak up.': 'Not Get Involved'
})

df['scenario_5_short'] = df['scenario_5'].map({
    'Move to zoom workshops': 'Move to Zoom',
    'Continue with in-person workshops': 'Continue In-Person'
})

sns.set_theme(style="whitegrid")

# Helper function to wrap text
def wrap_labels(ax, width, break_long_words=False):
    labels = []
    for label in ax.get_xticklabels():
        text = label.get_text()
        labels.append(textwrap.fill(text, width=width, break_long_words=break_long_words))
    ax.set_xticklabels(labels, rotation=45, ha='right')

# --- Plot 1: Horizontal Bar Chart for Resource Allocation by Major ---
fig, ax = plt.subplots(figsize=(14, 10))  # Correctly uses plt.subplots
major_data = df.groupby('major')['CE_Connection_Score'].mean().dropna().sort_values()

# Clean major names for display
major_labels = [textwrap.fill(str(m), 30) for m in major_data.index]

sns.barplot(x=major_data.values, y=major_labels, palette="viridis", ax=ax)
ax.set_title('Average Community Engagement Connection Score by Major', fontsize=16, pad=15, weight='bold')
ax.set_xlabel('Average CE Connection Score (Scale: 1 to 5)', fontsize=14)
ax.set_ylabel('')
ax.set_xlim(3.0, 5.0)

for p in ax.patches:
    ax.annotate(f"{p.get_width():.2f}", 
                (p.get_width() + 0.02, p.get_y() + p.get_height() / 2.), 
                ha='left', va='center', fontsize=11, weight='bold')

plt.tight_layout()
plt.savefig('major_resource_bar.png', dpi=300)
plt.close()

# --- Plot 2: Faceted Distribution Plot by Major ---
fig, axes = plt.subplots(1, 3, figsize=(24, 10)) # Correctly uses plt.subplots

# 1. CE Connection Score Distribution
sns.boxplot(x='major', y='CE_Connection_Score', data=df, ax=axes[0],
            color='white', linewidth=2, fliersize=0, boxprops=dict(alpha=0.5))
sns.swarmplot(x='major', y='CE_Connection_Score', data=df, ax=axes[0],
              palette='viridis', marker='h', size=8, hue='major', legend=False)
axes[0].set_title('CE Connection Score by Major', fontsize=16, pad=15, weight='bold')
axes[0].set_ylabel('CE Connection Score (1 to 5)', fontsize=14)
axes[0].set_xlabel('')
wrap_labels(axes[0], 20)

# 2. Scenario 2 Sentiment
sns.boxplot(x='major', y='scenario_2_reason_sentiment', hue='scenario_2_short', data=df, ax=axes[1],
            color='white', linewidth=1, fliersize=0, boxprops=dict(alpha=0.5))
sns.swarmplot(x='major', y='scenario_2_reason_sentiment', hue='scenario_2_short', data=df, ax=axes[1],
              palette='viridis', marker='h', size=7, dodge=True)
axes[1].set_title('Scenario 2 Sentiment by Major\n(Community Representation)', fontsize=16, pad=15, weight='bold')
axes[1].set_ylabel('Sentiment Score', fontsize=14)
axes[1].set_xlabel('')
axes[1].legend(title='Scenario 2 Decision', bbox_to_anchor=(1.05, 1), loc='upper left')
wrap_labels(axes[1], 20)

# 3. Scenario 5 Sentiment
sns.boxplot(x='major', y='scenario_5_reason_sentiment', hue='scenario_5_short', data=df, ax=axes[2],
            color='white', linewidth=1, fliersize=0, boxprops=dict(alpha=0.5))
sns.swarmplot(x='major', y='scenario_5_reason_sentiment', hue='scenario_5_short', data=df, ax=axes[2],
              palette='viridis', marker='h', size=7, dodge=True)
axes[2].set_title('Scenario 5 Sentiment by Major\n(Workshop Accessibility)', fontsize=16, pad=15, weight='bold')
axes[2].set_ylabel('Sentiment Score', fontsize=14)
axes[2].set_xlabel('')
axes[2].legend(title='Scenario 5 Decision', bbox_to_anchor=(1.05, 1), loc='upper left')
wrap_labels(axes[2], 20)

plt.tight_layout()
plt.savefig('refined_faceted_major.png', dpi=300)
plt.close()

/var/folders/cw/cx368b590mb_cg5tg17mvqv40000gn/T/ipykernel_20348/248786198.py:63: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=major_data.values, y=major_labels, palette="viridis", ax=ax)
/opt/anaconda3/lib/python3.12/site-packages/seaborn/categorical.py:3399: UserWarning: 33.3% of the points cannot be placed; you may want to decrease the size of the markers or use stripplot.
  warnings.warn(msg, UserWarning)
/var/folders/cw/cx368b590mb_cg5tg17mvqv40000gn/T/ipykernel_20348/248786198.py:54: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(labels, rotation=45, ha='right')
/var/folders/cw/cx368b590mb_cg5tg17mvqv40000gn/T/ipykernel_20348/248786198.py:92: FutureWarning: 

Setting a gradient palette using color= is deprecated and will be remo

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(24, 10))

# 1. CE Connection Score Distribution
sns.boxplot(x='major', y='CE_Connection_Score', data=df, ax=axes[0],
            color='white', linewidth=2, fliersize=0, boxprops=dict(alpha=0.5))
sns.swarmplot(x='major', y='CE_Connection_Score', data=df, ax=axes[0],
              palette='viridis', marker='h', size=8, hue='major', legend=False)
axes[0].set_title('CE Connection Score by Major', fontsize=16, pad=15, weight='bold')
axes[0].set_ylabel('CE Connection Score (1 to 5)', fontsize=14)
axes[0].set_xlabel('')
wrap_labels(axes[0], 20)

# 2. Scenario 2 Sentiment
sns.boxplot(x='major', y='scenario_2_reason_sentiment', hue='scenario_2_short', data=df, ax=axes[1],
            color='white', linewidth=1, fliersize=0, boxprops=dict(alpha=0.5))
sns.swarmplot(x='major', y='scenario_2_reason_sentiment', hue='scenario_2_short', data=df, ax=axes[1],
              palette='viridis', marker='h', size=7, dodge=True)
axes[1].set_title('Scenario 2 Sentiment by Major\n(Community Representation)', fontsize=16, pad=15, weight='bold')
axes[1].set_ylabel('Sentiment Score', fontsize=14)
axes[1].set_xlabel('')
axes[1].legend(title='Scenario 2 Decision', bbox_to_anchor=(1.05, 1), loc='upper left')
wrap_labels(axes[1], 20)

# 3. Scenario 5 Sentiment
sns.boxplot(x='major', y='scenario_5_reason_sentiment', hue='scenario_5_short', data=df, ax=axes[2],
            color='white', linewidth=1, fliersize=0, boxprops=dict(alpha=0.5))
sns.swarmplot(x='major', y='scenario_5_reason_sentiment', hue='scenario_5_short', data=df, ax=axes[2],
              palette='viridis', marker='h', size=7, dodge=True)
axes[2].set_title('Scenario 5 Sentiment by Major\n(Workshop Accessibility)', fontsize=16, pad=15, weight='bold')
axes[2].set_ylabel('Sentiment Score', fontsize=14)
axes[2].set_xlabel('')
axes[2].legend(title='Scenario 5 Decision', bbox_to_anchor=(1.05, 1), loc='upper left')
wrap_labels(axes[2], 20)

plt.tight_layout()
plt.savefig('refined_faceted_major.png', dpi=300)
plt.close()

/opt/anaconda3/lib/python3.12/site-packages/seaborn/categorical.py:3399: UserWarning: 33.3% of the points cannot be placed; you may want to decrease the size of the markers or use stripplot.
  warnings.warn(msg, UserWarning)
/var/folders/cw/cx368b590mb_cg5tg17mvqv40000gn/T/ipykernel_20348/248786198.py:54: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(labels, rotation=45, ha='right')
/var/folders/cw/cx368b590mb_cg5tg17mvqv40000gn/T/ipykernel_20348/525253895.py:14: FutureWarning: 

Setting a gradient palette using color= is deprecated and will be removed in v0.14.0. Set `palette='dark:white'` for the same effect.

  sns.boxplot(x='major', y='scenario_2_reason_sentiment', hue='scenario_2_short', data=df, ax=axes[1],
/opt/anaconda3/lib/python3.12/site-packages/seaborn/categorical.py:3399: UserWarning: 33.3% of the points cannot be placed; you may want to decrease the size of the markers o